In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import librosa
import itertools
import io
import numpy as np
import json

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
files = glob('cml-tts/*/*.parquet')
len(files)

1614

In [3]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'

    files, _ = files

    data = []
    for f in tqdm(files):
        base = '_'.join(f.split('/')[:2]) + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            try:
                t = df['text'].iloc[i].strip()
                if len(t) < 2:
                    continue
                audio_filename = f'{f_new}_{i}.mp3'
                audio_filename = os.path.join(base, audio_filename)
                b = df['audio'].iloc[i]['bytes']
                audio_np, sr = sf.read(io.BytesIO(b))
                if audio_np.ndim > 1:
                    audio_np = audio_np.mean(axis=1)
                if audio_np.shape[0] < 10000:
                    continue
                sf.write(audio_filename, audio_np, sr)
                
                data.append({
                    'audio_filename': audio_filename,
                    'text': df['text'].iloc[i],
                    'speaker': f"{base}_{df['speaker_id'].iloc[i]}"
                })
            except Exception as e:
                pass
        
    return data

In [4]:
# data = loop((files[:2], 0))

In [5]:
# len(data)

In [6]:
data = multiprocessing(files, loop, cores = min(50, len(files)))

100%|██████████| 14/14 [17:31<00:00, 75.10s/it]


In [9]:
len(data)

1336675

In [10]:
data[0]

{'audio_filename': 'cml-tts_german_audio/cml-tts-german-train-00344-of-00739-2a8af6d385f37941_0.mp3',
 'text': 'darum wird sie krank.',
 'speaker': 'cml-tts_german_audio_2037'}

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'cml-tts_german_audio/cml-tts-german-train-00344-of-00739-2a8af6d385f37941_0.mp3',
 'text': 'darum wird sie krank.',
 'speaker': 'cml-tts_german_audio_2037'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'cml-tts')

Creating parquet from Arrow format: 100%|██████████| 4/4 [00:00<00:00,  6.04ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  22%|██▏       | 27.2MB /  124MB,   ???B/s  
Processing Files (0 / 1):  94%|█████████▍|  116MB /  124MB,  443MB/s  
Processing Files (0 / 1):  99%|█████████▉|  123MB /  124MB,  239MB/s  
Processing Files (0 / 1): 100%|█████████▉|  123MB /  124MB,  120MB/s  
Processing Files (1 / 1): 100%|██████████|  124MB /  124MB, 98.9MB/s  
Processing Files (1 / 1): 100%|██████████|  124MB /  124MB, 96.4MB/s  
New Data Upload: 100%|██████████|  124MB /  124MB, 96.4MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.04s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/51472535ca24e32cdc4ca8d408af0797217fcaaf', commit_message='Upload dataset', commit_description='', oid='51472535ca24e32cdc4ca8d408af0797217fcaaf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [13]:
audio_files = [d['audio_filename'] for d in data]

with open('cml-tts-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [15]:
folders = glob('cml-tts_*_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

cml-tts_dutch_audio
cml-tts_german_audio
cml-tts_spanish_audio_neucodec
cml-tts_portuguese_audio_neucodec
cml-tts_polish_audio_neucodec
cml-tts_french_audio_neucodec
cml-tts_italian_audio_neucodec
cml-tts_polish_audio
cml-tts_german_audio_neucodec
cml-tts_spanish_audio
cml-tts_portuguese_audio
cml-tts_dutch_audio_neucodec
cml-tts_italian_audio
cml-tts_french_audio


In [20]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('cml-tts_*_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )